# Bangladesh Road Extraction - Colab

Give it a **latitude/longitude**, pick a **model**, get a **GeoJSON** of roads
you can drag into iD or JOSM.

Before you start: `Runtime` -> `Change runtime type` -> `T4 GPU` (free).
Not required, but much faster.

Run the cells top to bottom. Steps 0 and 1 only need to run once per session.

## Step 0 - Setup (run once)

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "No GPU - CPU mode, slower but works"

%cd /content
!rm -rf CNN
!git clone --depth 1 https://github.com/fsRakib/Road-Extraction.git CNN
%cd /content/CNN

# Colab already has numpy, scipy, pillow, requests, torch. These are missing:
!pip -q install rasterio shapely pyproj scikit-image segmentation-models-pytorch
print("\nSetup done.")

## Step 1 - Model weights (run once)

`dlinknet` is the model that actually works on Bangladesh roads.
`baseline` needs no weights. `unet` is included for comparison but is weak here.

In [ ]:
# D-LinkNet34 - DeepGlobe challenge winner (~119 MB)
!wget -q "https://www.dropbox.com/sh/h62vr320eiy57tt/AAB5Tm43-efmtYzW_GFyUCfma?dl=1" -O /tmp/dlink.zip
!unzip -o -q /tmp/dlink.zip -d /tmp/dlink
!cp /tmp/dlink/log01_dink34.th models/weights/dlinknet34_deepglobe.th

# U-Net - Massachusetts Roads Dataset (~120 MB)
!wget -q "https://huggingface.co/teohyc/Satellite-Road-Segmentation-UNet/resolve/main/best_road_seg_unet.pth" \
      -O models/weights/unet_road_massachusetts.pth

!ls -lh models/weights/

## Step 2 - Pick a location

Right-click a spot in Google Maps or OpenStreetMap and copy the coordinates.
Every tile is zoom 18, 2048x2048 px (~1.1 km across) - fixed in `config.py`
so model comparisons stay fair.

In [ ]:
#@markdown ### Enter coordinates
lat  = 23.846073      #@param {type:"number"}
lon  = 90.389624      #@param {type:"number"}
name = "airport_road" #@param {type:"string"}

!python download.py {lat} {lon} {name}

from IPython.display import Image, display
display(Image(f"data/images/{name}.png", width=500))

## Step 3 - Pick a model and extract roads

Available models (`baseline`, `dlinknet`, `unet`, `samroad`), or `all` to run every one:

In [ ]:
!python extract.py list

In [ ]:
#@markdown ### Choose a model
model = "dlinknet" #@param ["baseline", "dlinknet", "unet", "all"]

!python extract.py {name} {model}

from IPython.display import Image, display
for m in (["baseline", "dlinknet", "unet"] if model == "all" else [model]):
    print(f"--- {m} ---")
    display(Image(f"outputs/images/{name}_{m}.png", width=650))

## Step 4 - Download the GeoJSON

Then in iD: navigate to your coordinates first, press `F` -> Custom Map Data ->
drop the file on the map. Trace over it by hand - **never bulk-upload model
output to OSM**, it violates the Automated Edits policy.

In [ ]:
from google.colab import files
for m in (["baseline", "dlinknet", "unet"] if model == "all" else [model]):
    files.download(f"outputs/geojson/{name}_{m}.geojson")

---
## Optional - SAM-Road (GPU strongly recommended)

Outputs a road *graph* instead of a mask. In this project's own testing it did
noticeably worse than `dlinknet` on Bangladesh rural roads - see the docstring in
`models/samroad.py`. Skip this unless you specifically want to see it.

Downloads ~1.4 GB of checkpoints.

In [ ]:
!git clone --depth 1 https://github.com/htcr/sam_road.git /root/sam_road
!git clone --depth 1 https://github.com/htcr/segment-anything-road.git /root/sam_road/sam
!pip -q install lightning pytorch_lightning wandb rtree imageio opencv-python-headless \
                torchmetrics addict tcod scikit-learn python-igraph networkx matplotlib

!mkdir -p /root/sam_road/sam_ckpts
!wget -q -O /root/sam_road/sam_ckpts/sam_vit_b_01ec64.pth \
      https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!wget -q -O models/weights/cityscale_vitb_512_e10.ckpt \
      https://huggingface.co/congrui/sam_road/resolve/main/cityscale_vitb_512_e10.ckpt

In [ ]:
import os
os.environ["SAM_ROAD_DIR"] = "/root/sam_road"
os.environ["SAMROAD_PATCHES_PER_EDGE"] = "16"   # drop to "6" if you have no GPU

!python extract.py {name} samroad

from IPython.display import Image, display
display(Image(f"outputs/images/{name}_samroad.png", width=650))

from google.colab import files
files.download(f"outputs/geojson/{name}_samroad.geojson")

---
## Notes

- **Adding a model:** drop a file in `models/` following `models/_base.py`.
  `core/registry.py` picks it up automatically and it shows in `extract.py list`.
- **Editing project code:** this notebook clones from GitHub, so push your changes
  first (`git push`), then re-run Step 0.
- **Validating against OSM** (`validate.py`) needs `bangladesh-260823.osm.pbf` (~350 MB).
  Mount Google Drive rather than using the browser upload widget.